In [ ]:
import mlflow.spark
from mlflow.models.signature import infer_signature
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.types import StringType, DoubleType, IntegerType, FloatType, DecimalType
import mlflow

In [ ]:

model_path = "/Volumes/nff_catalog_f586_dev/gold_ml/model/churn.model"

def train_churn_model(table_name = "nff_catalog_f586_dev.gold_ml.churn_features",
                      experiment_name="/Shared/churn_model_experiment",
                      run_name="churn_rf_pipeline"):
    
    # 1. Setup MLflow & Autologging
    mlflow.set_experiment(experiment_name)
    mlflow.pyspark.ml.autolog(log_models=False) # We log manually to include signatures

    # 2. Load Data
    df = spark.read.table(table_name)
    train_data, test_data = df.randomSplit([0.7, 0.3], seed=42)
    
    # Cache for performance since we count and train
    train_data.cache()
    test_data.cache()

    # 3. Feature Engineering Stages
    categorical_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType) and f.name != "is_churned"]
    
    # Batch processing for Indexer and Encoder (Spark 3.0+)
    indexer = StringIndexer(
        inputCols=categorical_cols, 
        outputCols=[c + '_index' for c in categorical_cols], 
        handleInvalid="keep"
    )
    encoder = OneHotEncoder(
        inputCols=indexer.getOutputCols(), 
        outputCols=[c + '_vec' for c in categorical_cols]
    )

    numerical_types = (DoubleType, IntegerType, FloatType, DecimalType)
    numerical_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, numerical_types) and f.name != "is_churned"]
    
    # Assemble all features
    all_feature_names = numerical_cols + [c + '_vec' for c in categorical_cols]
    assembler = VectorAssembler(inputCols=all_feature_names, outputCol="raw_features")
    scaler = StandardScaler(inputCol="raw_features", outputCol="features")

    # 4. Model Definition
    rf = RandomForestClassifier(labelCol="is_churned", featuresCol="features", numTrees=100, seed=42)

    # 5. Pipeline Construction
    pipeline = Pipeline(stages=[indexer, encoder, assembler, scaler, rf])

    with mlflow.start_run(run_name=run_name) as run:
        # Train
        model = pipeline.fit(train_data)
        
        # Predict & Evaluate
        predictions = model.transform(test_data)
        evaluator = BinaryClassificationEvaluator(labelCol="is_churned", metricName="areaUnderROC")
        auc = evaluator.evaluate(predictions)

        # Log Manual Params & Metrics
        mlflow.log_param("table_name", table_name)
        mlflow.log_metric("auc", auc)

        # 6. Feature Importance Logging
        rf_stage = model.stages[-1]
        importances = rf_stage.featureImportances.toArray()
        # Note: Mapping these to names is approximate due to OneHot vector expansion
        mlflow.log_dict({"feature_importances": dict(zip(all_feature_names, importances.tolist()))}, "feature_importance.json")

        # 7. Signature and Model Logging
        # Use a small sample for the signature/example
        input_example = train_data.limit(5).toPandas()
        signature = infer_signature(train_data.drop("is_churned"), predictions.select("prediction"))
        
        mlflow.spark.log_model(
            spark_model=model,
            artifact_path="churn_model",
            signature=signature,
            input_example=input_example
        )

        print("-" * 30)
        print(f"✅ Training Complete. AUC: {auc:.4f}")
        print(f"Run ID: {run.info.run_id}")
        print("-" * 30)
    
    # Clean up cache
    train_data.unpersist()
    test_data.unpersist()

    return model

In [ ]:
model = train_churn_model()